# SOEN Disruption Prediction — PCA-8 Per-Timestep Binary Classification

**Task**: `seq2seq` with `cross_entropy` — per-timestep prediction at each of 279 SOEN steps,
directly comparable to the TCN baseline's per-timestep BCE.

**Hardware mapping**: 28 dendrite neurons × 8 synaptic inputs = 224 inputs per step.
7812 raw timesteps → 279 SOEN steps (group every 28). Labels sampled at the **last**
of each 28-step group from the original per-timestep target.

**Architecture**: `Linear(224) → SingleDendrite(28, rec) → SQUID(24) → Linear(2)`

**Balancing**: Training set oversampled at sequence level (50/50 disruptive/clear)
to match TCN's stratified batch sampler. Target mask excludes post-disruption steps.

**Deliverable**: Quantized `.soen` checkpoint for hardware deployment.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

import numpy as np
import h5py
import yaml

# ── Find soen_toolkit src ─────────────────────────────────────────
_SOEN_SRC_CANDIDATES = [
    Path("/home/idies/workspace/Temporary/dpark1/scratch/soenhardware/soen-toolkit/src"),
    Path("/home/idies/workspace/Temporary/dpark1/scratch/SOEN/soenre2/src"),
    Path("/Users/davidpark/Documents/Cursor/soenhardware/soen-toolkit/src"),
]

SOEN_SRC = None
for _c in _SOEN_SRC_CANDIDATES:
    if (_c / "soen_toolkit" / "__init__.py").exists():
        SOEN_SRC = _c
        break
if SOEN_SRC is None:
    raise FileNotFoundError("soen_toolkit src not found")

TUTORIAL_DIR = SOEN_SRC / "soen_toolkit" / "tutorial_notebooks" / "time_to_event_tutorial"
print(f"soen_toolkit src: {SOEN_SRC}")

# ── Find a Python interpreter that can actually import soen_toolkit ──
# The Jupyter kernel may be Python 3.9 which can't handle `str | Path` syntax.
# Search for uv-managed or conda Python ≥3.10 that works.
_PYTHON_CANDIDATES = [
    # uv-managed venv in the soen-toolkit project
    SOEN_SRC.parent / ".venv" / "bin" / "python",
    SOEN_SRC.parent / ".venv" / "bin" / "python3",
    # conda envs
    Path("/home/idies/workspace/Temporary/dpark1/scratch/conda/conda_envs/soen/bin/python"),
    # system
    shutil.which("python3.11") or "",
    shutil.which("python3.10") or "",
    sys.executable,  # fallback to kernel python
]

SOEN_PYTHON = None
for _p in _PYTHON_CANDIDATES:
    _p = Path(str(_p))
    if not _p.exists():
        continue
    # Test if this python can import soen_toolkit
    _test = subprocess.run(
        [str(_p), "-c", "import soen_toolkit; print('OK')"],
        env={**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)},
        capture_output=True, text=True,
    )
    if _test.returncode == 0 and "OK" in _test.stdout:
        SOEN_PYTHON = _p
        break

if SOEN_PYTHON is None:
    raise RuntimeError(
        "No Python interpreter can import soen_toolkit. Tried:\n" +
        "\n".join(f"  {p}" for p in _PYTHON_CANDIDATES if Path(str(p)).exists())
    )

# Get Python version
_ver = subprocess.run([str(SOEN_PYTHON), "--version"], capture_output=True, text=True)
print(f"SOEN Python:      {SOEN_PYTHON} ({_ver.stdout.strip()})")
print(f"Kernel Python:    {sys.executable} (Python {sys.version.split()[0]})")

# ── Helper: run code in the soen-compatible Python ────────────────
def run_soen_python(code: str, check: bool = True) -> subprocess.CompletedProcess:
    """Run Python code using the soen-compatible interpreter."""
    env = {**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)}
    return subprocess.run(
        [str(SOEN_PYTHON), "-c", code],
        env=env, check=check, capture_output=True, text=True,
    )

# ── Paths ─────────────────────────────────────────────────────────
PCA8_H5 = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/pca8_100x_flattop/all_data.h5")

SOEN_DIR = Path("soen_training")
DATASET_DIR = SOEN_DIR / "datasets"
MODEL_DIR = SOEN_DIR / "model_specs"
CONFIG_DIR = SOEN_DIR / "training_configs"
RESULTS_DIR = SOEN_DIR / "results"

for d in [DATASET_DIR, MODEL_DIR, CONFIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PCA8 source:      {PCA8_H5}")
print(f"SOEN dir:         {SOEN_DIR.resolve()}")

## 1. Convert PCA8 H5 → SOEN HDF5 format (seq2seq per-timestep)

Reshape `(N, 7812, 8)` → `(N, 279, 224)`. For labels and weights, take the
**last of each 28-step group** from the original per-timestep arrays.

Training set is oversampled at the sequence level (50/50 disruptive/clear).

In [ ]:
SOEN_H5 = DATASET_DIR / "pca8_disruption_seq2seq_2class.h5"
N_NEURONS = 28   # SOEN hidden neurons
N_INPUTS = 8     # synaptic inputs per neuron (PCA components)
INPUT_DIM = N_NEURONS * N_INPUTS  # 224


def reshape_for_soen(X_ch_first):
    """(N, 8, 7812) → (N, 279, 224): group 28 timesteps into 28×8 input vector."""
    X = np.transpose(X_ch_first, (0, 2, 1))        # (N, 7812, 8)
    N, T, D = X.shape
    T_steps = T // N_NEURONS                         # 279
    X = X[:, :T_steps * N_NEURONS, :]                # trim to exact multiple
    return X.reshape(N, T_steps, N_NEURONS * D), T_steps


def downsample_labels(arr_1d, T_steps):
    """Take last of each 28-step group: (T,) → (T_steps,)."""
    return arr_1d[:T_steps * N_NEURONS].reshape(T_steps, N_NEURONS)[:, -1]


def balance_sequences(X, labels, target_mask, seq_labels):
    """Oversample minority class at sequence level to 50/50."""
    pos_idx = np.where(seq_labels == 1)[0]
    neg_idx = np.where(seq_labels == 0)[0]
    n_pos, n_neg = len(pos_idx), len(neg_idx)
    if n_pos == n_neg or n_pos == 0 or n_neg == 0:
        return X, labels, target_mask

    rng = np.random.default_rng(42)
    if n_pos < n_neg:
        extra = rng.choice(pos_idx, size=n_neg - n_pos, replace=True)
    else:
        extra = rng.choice(neg_idx, size=n_pos - n_neg, replace=True)
    all_idx = np.concatenate([np.arange(len(seq_labels)), extra])
    rng.shuffle(all_idx)
    return X[all_idx], labels[all_idx], target_mask[all_idx]


with h5py.File(PCA8_H5, "r") as src, h5py.File(SOEN_H5, "w") as dst:
    for split in ("train", "val", "test"):
        # Reshape input
        X, T_steps = reshape_for_soen(np.asarray(src[f"{split}/X"]))
        N = X.shape[0]

        # Per-timestep labels: last of each 28-group from original target
        raw_target = np.asarray(src[f"{split}/target"])    # (N, 7812)
        labels = np.stack([downsample_labels(raw_target[i], T_steps) for i in range(N)])
        labels = labels.astype(np.int64)                    # (N, 279)

        # Target mask: last of each 28-group from original weight
        raw_weight = np.asarray(src[f"{split}/weight"])    # (N, 7812)
        target_mask = np.stack([downsample_labels(raw_weight[i], T_steps) for i in range(N)])
        target_mask = (target_mask > 0).astype(bool)        # (N, 279)

        # Sequence-level labels for balancing
        seq_labels = np.asarray(src[f"{split}/labels"])     # (N,) int64

        # Balance training set (oversample minority at sequence level)
        if split == "train":
            n_before = N
            n_pos_before = int((seq_labels == 1).sum())
            X, labels, target_mask = balance_sequences(X, labels, target_mask, seq_labels)
            N = X.shape[0]
            print(f"  {split}: {n_before} ({n_pos_before} disruptive) → {N} (balanced 50/50)")

        g = dst.create_group(split)
        g.create_dataset("data", data=X, dtype=np.float32)
        g.create_dataset("labels", data=labels, dtype=np.int64)
        g.create_dataset("target_mask", data=target_mask)
        g.create_dataset("input_mask", data=np.ones(X.shape[:2], dtype=bool))

        n_pos_ts = int(labels.sum())
        n_valid_ts = int(target_mask.sum())
        n_pos_seq = int(labels.any(axis=1).sum())
        print(f"  {split}: N={N}, T={T_steps}, D={X.shape[2]}, "
              f"seq_disruptive={n_pos_seq}/{N}, "
              f"ts_positive={n_pos_ts}/{n_valid_ts} valid")

print(f"\nSaved: {SOEN_H5}")
print(f"  Input shape: ({T_steps}, {INPUT_DIM}), Labels: ({T_steps},) per sample")

## 2. Build SOEN model (block-diagonal input + SQUID readout)

`Linear(224) →[block_diagonal(28)]→ SingleDendrite(28, rec) → SQUID(24) → Linear(2)`

**J_0_to_1**: `block_diagonal(num_heads=28)` — neuron i only sees inputs `[i*8:(i+1)*8]`.
This is the correct hardware mapping: each of 28 dendrites has exactly 8 synaptic inputs.

Cross-entropy with 2 output classes is functionally equivalent to BCE.

In [ ]:
MODEL_PATH = MODEL_DIR / "224IN_28H_24SQUID_2OffchipLinear_blockdiag.soen"

# Build model with block_diagonal input connection (each neuron gets its own 8 inputs)
_build_code = f"""
from pathlib import Path
from soen_toolkit import nn

model_path = Path({str(MODEL_PATH.resolve())!r})
model_path.parent.mkdir(parents=True, exist_ok=True)

g = nn.Graph(dt=10.0, network_evaluation_method="layerwise")

# Layer 0: Input (224 = 28 neurons × 8 PCA inputs)
g.add_layer(0, nn.layers.Linear(dim=224), description="input")

# Layer 1: Hidden — 28 SingleDendrite neurons (recurrent)
g.add_layer(1, nn.layers.SingleDendrite(
    dim=28,
    solver="FE",
    source_func_type="RateArray",
    phi_offset=nn.param_specs.constant(0.23, learnable=False, sharing_mode="shared",
                                        constraints={{"min": -0.5, "max": 0.5}}),
    bias_current=nn.param_specs.constant(1.7, learnable=False, sharing_mode="shared",
                                          constraints={{"min": 1.1, "max": 2.0}}),
    gamma_plus=nn.param_specs.constant(2.3508e-5, learnable=False, sharing_mode="shared"),
    gamma_minus=nn.param_specs.constant(2.6995e-5, learnable=False, sharing_mode="shared",
                                         constraints={{"min": 1.7997e-5, "max": 5.3991e-5}}),
), description="hidden")

# Layer 2: SQUID readout — first 24 of 28 neurons
g.add_layer(2, nn.layers.DendriteReadout(
    dim=24,
    phi_offset=nn.param_specs.constant(0.23, learnable=False, sharing_mode="shared",
                                        constraints={{"min": -0.5, "max": 0.5}}),
    bias_current=nn.param_specs.constant(1.7, learnable=False, sharing_mode="shared",
                                          constraints={{"min": 1.1, "max": 2.0}}),
), description="readout_squid")

# Layer 3: Off-chip linear projection → 2 classes
g.add_layer(3, nn.layers.Linear(dim=2), description="offchip_projection")

# ── Connections ──

# J_0_to_1: BLOCK DIAGONAL — neuron i receives only inputs [i*8:(i+1)*8]
# 224 inputs / 28 neurons = 8 inputs per neuron (correct hardware mapping)
g.connect(0, 1,
    structure=nn.structure.block_diagonal(num_heads=28),
    init=nn.init.xavier_uniform(gain=1.0),
    constraints={{"min": -0.14, "max": 0.14}},
    learnable=True,
)

# J_1_to_1: Dense recurrent (28→28)
g.connect(1, 1,
    structure=nn.structure.dense(),
    init=nn.init.orthogonal(gain=0.1),
    constraints={{"min": -0.14, "max": 0.14}},
    learnable=True,
)

# J_1_to_2: One-to-one, first 24 neurons → SQUID, fixed J=0.5
g.connect(1, 2,
    structure=nn.structure.one_to_one(source_start_node_id=0, source_end_node_id=23),
    init=nn.init.constant(0.5),
    learnable=False,
)

# J_2_to_3: Off-chip float projection (24→2), NOT quantized
g.connect(2, 3,
    structure=nn.structure.dense(),
    init=nn.init.xavier_uniform(gain=1.0),
    learnable=True,
)

g.compile()
g.save(str(model_path))
print("OK")
"""

_r = run_soen_python(_build_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Model build failed")

print(f"Model saved: {MODEL_PATH}")
print(f"  Layer 0: Linear(224)             — input (28×8)")
print(f"  Layer 1: SingleDendrite(28)      — hidden, recurrent")
print(f"      J_0→1: block_diagonal(28)    — neuron i gets inputs [i*8:(i+1)*8]")
print(f"      J_1→1: dense(28,28)          — recurrent")
print(f"  Layer 2: DendriteReadout(24)     — SQUID, first 24 neurons")
print(f"      J_1→2: one_to_one, J=0.5     — fixed")
print(f"  Layer 3: Linear(2)               — off-chip → softmax")
print(f"      J_2→3: dense(24,2)           — trainable float")
print(f"  File exists: {MODEL_PATH.exists()}")

## 3. Build training config

`seq2seq` + `cross_entropy` per-timestep, with `label_mask_key: target_mask`
to exclude post-disruption timesteps from loss (same as TCN's weight masking).

In [ ]:
# ── Run settings ──
EXPERIMENT_NAME = "pca8_disruption_seq2seq_2class"
MAX_EPOCHS = 200
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
BACKEND = "jax"
DT_NS = 10.0
READOUT_VARIANT = "offchip_linear"
NUM_CLASSES = 2

CACHE_STRATEGY = "full"

CONFIG_PATH = CONFIG_DIR / f"training_config_{EXPERIMENT_NAME}.yaml"
BASE_CONFIG_PATH = TUTORIAL_DIR / "training" / "training_configs" / "bnl_tutorial_base.yaml"

# Build config: use build_training_config_from_knobs for the base,
# then patch to seq2seq + cross_entropy + target_mask
_config_code = f"""
import sys, json, yaml
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from pathlib import Path
from notebook_utils import build_training_config_from_knobs

# Start with seq2seq_regression to get seq2seq mapping (no time_pooling)
profile = build_training_config_from_knobs(
    base_config_path=Path({str(BASE_CONFIG_PATH.resolve())!r}),
    out_config_path=Path({str(CONFIG_PATH.resolve())!r}),
    dataset_path=Path({str(SOEN_H5.resolve())!r}),
    model_path=Path({str(MODEL_PATH.resolve())!r}),
    task_type='seq2seq_regression',
    experiment_name={EXPERIMENT_NAME!r},
    max_epochs={MAX_EPOCHS},
    batch_size={BATCH_SIZE},
    learning_rate={LEARNING_RATE},
    backend={BACKEND!r},
    num_classes=0,
    dt_ns={DT_NS},
    readout_variant={READOUT_VARIANT!r},
    use_minmax_input_scaling=True,
    input_scale_min=0.0,
    input_scale_max=1.0,
    cache_strategy={CACHE_STRATEGY!r},
)

# Patch: switch from MSE regression to cross_entropy classification
cfg_path = Path({str(CONFIG_PATH.resolve())!r})
cfg = yaml.safe_load(cfg_path.read_text())

cfg["training"]["losses"] = [{{"name": "cross_entropy", "weight": 1.0}}]
cfg["data"]["num_classes"] = {NUM_CLASSES}
cfg["data"]["label_mask_key"] = "target_mask"   # mask post-disruption timesteps

# Only use accuracy for seq2seq — f1/precision/recall are seq2static-only in JAX backend
cfg["logging"]["metrics"] = ["accuracy"]
cfg["logging"]["log_batch_metrics"] = True

with cfg_path.open("w") as f:
    yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

print(json.dumps(profile, default=str))
"""

_r = run_soen_python(_config_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Config build failed")

import json as _json
_profile = _json.loads(_r.stdout.strip().split("\n")[-1])

# Verify
_cfg_check = yaml.safe_load(CONFIG_PATH.read_text())
print(f"Config saved: {CONFIG_PATH}")
print(f"  mapping:         {_cfg_check['training']['mapping']}")
print(f"  losses:          {_cfg_check['training']['losses']}")
print(f"  num_classes:     {_cfg_check['data']['num_classes']}")
print(f"  label_mask_key:  {_cfg_check['data'].get('label_mask_key')}")
print(f"  total_time_ns:   {_cfg_check['data'].get('total_time_ns')}")
print(f"  sequence_length: {_cfg_check['data'].get('sequence_length')}")
print(f"  metrics:         {_cfg_check['logging']['metrics']}")
print(f"  cache:           {_cfg_check['data'].get('cache')}")
print(f"  Dataset profile: {_profile}")

## 4. Train

Invokes `soen_toolkit.training` via subprocess (same as tutorial `02_train_models.ipynb`).
Saves `initial.soen` and `last.soen` checkpoints in `.soen` format.

In [ ]:
# Training in chunks of EVAL_EVERY epochs, with F1 evaluation between chunks
import os as _os
import json as _json
import re as _re
import glob as _glob

EVAL_EVERY = 10

print(f"Training: {MAX_EPOCHS} epochs in chunks of {EVAL_EVERY}, with F1 eval between chunks")
print(f"  Python: {SOEN_PYTHON}")
print(f"  Config: {CONFIG_PATH}")
print()


def find_last_soen():
    """Find the most recent last.soen checkpoint across all possible results dirs."""
    search_roots = [
        str(RESULTS_DIR.resolve()),
        str(TUTORIAL_DIR / "training" / "results"),
    ]
    candidates = []
    for root in search_roots:
        for p in _glob.glob(f"{root}/**/last.soen", recursive=True):
            if EXPERIMENT_NAME in p:
                candidates.append(p)
    if not candidates:
        return None
    return max(candidates, key=lambda p: _os.path.getmtime(p))


def eval_f1(split="val"):
    """Load last.soen, run inference, compute masked F1."""
    last_soen = find_last_soen()
    if last_soen is None:
        return None, "", "last.soen not found in any results directory"

    code = f"""
import sys, json, numpy as np
import h5py, torch
from soen_toolkit.core.soen_model_core import SOENModelCore

model = SOENModelCore.load({last_soen!r})
model.eval()

results = {{}}
for sp in ["train", "{split}"]:
    with h5py.File({str(SOEN_H5.resolve())!r}, "r") as f:
        X = torch.from_numpy(np.asarray(f[sp]["data"]))
        labels = np.asarray(f[sp]["labels"])       # (N, 279)
        mask = np.asarray(f[sp]["target_mask"])     # (N, 279)
    T_label = labels.shape[1]  # 279
    preds_list = []
    with torch.no_grad():
        for i in range(0, len(X), 64):
            out = model(X[i:i+64])
            if isinstance(out, tuple):
                out = out[0]
            # Model may output T+1 steps (initial state); trim to match labels
            out_np = out.argmax(dim=-1).cpu().numpy()  # (B, T_out)
            if out_np.shape[1] > T_label:
                out_np = out_np[:, -T_label:]  # keep last T_label steps
            elif out_np.shape[1] < T_label:
                out_np = out_np[:, :T_label]
            preds_list.append(out_np)
    preds = np.concatenate(preds_list)  # (N, 279)
    valid = mask.astype(bool)
    p, l = preds[valid], labels[valid]
    tp = int(((p==1)&(l==1)).sum())
    fp = int(((p==1)&(l==0)).sum())
    fn = int(((p==0)&(l==1)).sum())
    prec = tp/(tp+fp+1e-8)
    rec = tp/(tp+fn+1e-8)
    f1 = 2*prec*rec/(prec+rec+1e-8)
    acc = float((p==l).mean())
    results[sp] = {{"acc": round(acc,4), "f1": round(f1,4),
                    "prec": round(prec,4), "rec": round(rec,4)}}
print(json.dumps(results))
"""
    r = run_soen_python(code, check=False)
    if r.returncode == 0 and r.stdout.strip():
        try:
            return _json.loads(r.stdout.strip().split("\n")[-1])
        except _json.JSONDecodeError:
            return None, r.stdout[-300:], r.stderr[-300:]
    return None, "", r.stderr[-500:] if r.stderr else ""


# ── Training loop ──
import yaml as _yaml

for chunk_end in range(EVAL_EVERY, MAX_EPOCHS + 1, EVAL_EVERY):
    cfg = _yaml.safe_load(CONFIG_PATH.read_text())
    cfg["training"]["max_epochs"] = chunk_end
    with CONFIG_PATH.open("w") as f:
        _yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

    result = subprocess.run(
        [str(SOEN_PYTHON), "-m", "soen_toolkit.training", str(CONFIG_PATH.resolve())],
        env={**_os.environ, "PYTHONPATH": str(SOEN_SRC)},
    )

    if result.returncode != 0:
        print(f"\n  Training failed at epoch {chunk_end}")
        break

    ret = eval_f1("val")
    if isinstance(ret, tuple):
        metrics, stdout_tail, stderr_tail = None, ret[1], ret[2]
    else:
        metrics = ret

    print(f"\n{'─'*80}")
    print(f"  EPOCH {chunk_end} F1 EVALUATION")
    print(f"{'─'*80}")
    if metrics:
        for sp in ["train", "val"]:
            m = metrics.get(sp, {})
            print(f"  {sp:>5s}:  acc={m.get('acc','N/A'):>7}  f1={m.get('f1','N/A'):>7}  "
                  f"prec={m.get('prec','N/A'):>7}  rec={m.get('rec','N/A'):>7}")
    else:
        print(f"  F1 eval failed")
        if stderr_tail:
            print(f"  stderr: ...{stderr_tail[-300:]}")
    print(f"{'─'*80}\n")

print("\nTraining complete.")

## 5. Plot loss curves

In [ ]:
import matplotlib.pyplot as plt
import json as _json

# Read TensorBoard scalars via subprocess
_tb_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from notebook_utils import read_training_scalars_with_fallback
scalars = read_training_scalars_with_fallback({str(CONFIG_PATH.resolve())!r})
out = {{}}
for tag, df in scalars.items():
    out[tag] = {{"step": df["step"].tolist(), "value": df["value"].tolist()}}
print(json.dumps(out))
"""
_r = run_soen_python(_tb_code, check=False)
scalars = {}
if _r.returncode == 0 and _r.stdout.strip():
    try:
        raw = _json.loads(_r.stdout.strip().split("\n")[-1])
        scalars = {k: v for k, v in raw.items()}
    except _json.JSONDecodeError:
        print("Warning: could not parse TensorBoard output")
else:
    print(f"Warning: TensorBoard read failed. stderr: {_r.stderr[:500] if _r.stderr else 'none'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for tag, data in scalars.items():
    if "loss" in tag.lower() and "epoch" in tag.lower():
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(data["step"], data["value"], label=label)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
found_acc = False
for tag, data in scalars.items():
    if "acc" in tag.lower() and "epoch" in tag.lower():
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(data["step"], data["value"], label=label)
        found_acc = True
if found_acc:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Per-timestep Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No accuracy metrics logged", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

## 6. Audit trained weights and quantize to 3-bit `.soen`

Verify:
1. All trainable connections (J_0_to_1, J_1_to_1) are within [-0.14, 0.14]
2. Fixed parameters (phi_offset, bias_current, gamma, J_1_to_2) are unchanged
3. QAT produced valid 3-bit (9-level) quantized weights

**Output**: `last_quant_3bit9lvl.soen` — the hardware deliverable.

In [ ]:
import json as _json

# Audit and quantize via subprocess
_audit_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from pathlib import Path
from notebook_utils import audit_and_quantize_latest_run

audit = audit_and_quantize_latest_run(
    results_dir=Path({str(RESULTS_DIR.resolve())!r}),
    config_path=Path({str(CONFIG_PATH.resolve())!r}),
    experiment_name={EXPERIMENT_NAME!r},
    target_connections=["J_0_to_1", "J_1_to_1"],
    weight_min=-0.14,
    weight_max=0.14,
    quant_levels=9,
)

# Serialize for transfer back to notebook
out = {{
    "bounds_ok": audit["bounds_ok"],
    "fixed_params_unchanged": audit["fixed_params_unchanged"],
    "qat_active_in_config": audit["qat_active_in_config"],
    "bounds_stats": {{k: {{kk: float(vv) if isinstance(vv, (int, float)) else vv for kk, vv in v.items()}} for k, v in audit["bounds_stats"].items()}},
    "quant_levels_present": {{k: int(v) for k, v in audit.get("quant_levels_present", {{}}).items()}},
    "checkpoint_dir": str(audit["checkpoint_dir"]),
    "quantized_checkpoint": str(audit["quantized_checkpoint"]),
}}
loss_info = audit.get("loss_info", {{}})
out["loss_info"] = {{k: (float(v) if isinstance(v, (int, float)) else str(v)) for k, v in loss_info.items()}}
print(json.dumps(out))
"""

_r = run_soen_python(_audit_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Audit failed")

audit = _json.loads(_r.stdout.strip().split("\n")[-1])

print("=== Audit Results ===")
print(f"  Bounds OK:              {audit['bounds_ok']}")
print(f"  Fixed params unchanged: {audit['fixed_params_unchanged']}")
print(f"  QAT active in config:   {audit['qat_active_in_config']}")

print("\n=== Weight Bounds ===")
for conn, stats in audit["bounds_stats"].items():
    print(f"  {conn}: min={stats['min']:.6f}, max={stats['max']:.6f}, in_bounds={stats['in_bounds']}")

print("\n=== Quantization Levels ===")
for conn, n_lvl in audit.get("quant_levels_present", {}).items():
    print(f"  {conn}: {n_lvl} unique quantized values")

print("\n=== Loss (float vs quantized) ===")
loss_info = audit.get("loss_info", {})
print(f"  Train loss (float32):   {loss_info.get('train_loss_float', 'N/A')}")
print(f"  Train loss (quantized): {loss_info.get('train_loss_quantized', 'N/A')}")

print(f"\n=== Hardware Deliverable ===")
print(f"  {audit['quantized_checkpoint']}")

## 7. Verify checkpoint structure

Load the quantized `.soen` file and inspect its contents to confirm it matches the expected format for hardware deployment.

In [ ]:
# Inspect the quantized checkpoint via subprocess
_inspect_code = f"""
import sys, json, torch
sys.path.insert(0, {str(SOEN_SRC)!r})

quant_path = {str(Path(audit['quantized_checkpoint']))!r}
obj = torch.load(quant_path, map_location="cpu", weights_only=False)

out = {{"keys": list(obj.keys()), "model_type": obj.get("model_type", "N/A"), "dt_ns": obj.get("dt_ns", "N/A")}}

sd = obj.get("state_dict", {{}})
sd_info = {{}}
for k, v in sd.items():
    if hasattr(v, "shape"):
        sd_info[k] = {{"shape": list(v.shape), "dtype": str(v.dtype)}}
    else:
        sd_info[k] = {{"value": str(v)}}
out["state_dict"] = sd_info

layers = []
for lc in obj.get("layers_config", []):
    layers.append({{"id": lc.get("id"), "type": lc.get("type"), "dim": lc.get("dim"), "desc": lc.get("description", "")}})
out["layers"] = layers

conns = []
for cc in obj.get("connections_config", []):
    conns.append({{"src": cc.get("source_layer_id"), "tgt": cc.get("target_layer_id"),
                  "structure": cc.get("structure", {{}}).get("type", "?"), "learnable": cc.get("learnable")}})
out["connections"] = conns

print(json.dumps(out))
"""

_r = run_soen_python(_inspect_code, check=False)
if _r.returncode == 0:
    info = _json.loads(_r.stdout.strip().split("\n")[-1])

    print(f"=== .soen checkpoint ===")
    print(f"Top-level keys: {info['keys']}")
    print(f"model_type: {info['model_type']}")
    print(f"dt_ns:      {info['dt_ns']}")

    print(f"\n=== state_dict ===")
    for k, v in info["state_dict"].items():
        if "shape" in v:
            print(f"  {k}: shape={v['shape']}, dtype={v['dtype']}")
        else:
            print(f"  {k}: {v['value']}")

    print(f"\n=== Layers ===")
    for l in info["layers"]:
        print(f"  Layer {l['id']}: {l['type']} dim={l['dim']} — {l['desc']}")

    print(f"\n=== Connections ===")
    for c in info["connections"]:
        print(f"  {c['src']}→{c['tgt']}: {c['structure']}, learnable={c['learnable']}")
else:
    print("Checkpoint inspection failed:")
    print(_r.stderr)